# LlamaIndex Workflow: Multi-Step Data Tasks with Context Retrieval
### Structured Pipelines · Stateful Context · Validated Outputs

**Project -- NLP / Retrieval-Augmented Generation Portfolio**

This notebook implements a **LlamaIndex-style Workflow** that chains multi-step data tasks together with:
- **Context retrieval** between steps (each step reads results from previous steps)
- **Structured outputs** with Pydantic-validated schemas
- **Stateful event passing** via a typed event bus
- **Parallel step execution** for independent work
- **Step-level retry and fallback** logic
- **Observability** through a built-in tracing layer

The problem domain is an **automated business intelligence pipeline** that ingests raw sales data, enriches it via document retrieval, generates analyst notes, and produces a formatted executive report.

## What is a LlamaIndex Workflow?

LlamaIndex's `Workflow` abstraction (introduced in v0.10) solves a common problem: **chaining LLM-powered steps while keeping state, retrying on failure, and running independent steps in parallel.**

### Traditional Pipeline vs Workflow

| Feature | Simple Pipeline | LlamaIndex Workflow |
|---|---|---|
| Step communication | Return value → next input | Typed events via event bus |
| Parallelism | Sequential only | Parallel steps on same event |
| State sharing | Pass large context dicts | `Context` object shared across steps |
| Error handling | Try/except per step | Per-step retry with configurable policy |
| Observability | Print statements | Built-in span tracing |
| Output validation | Manual | Pydantic model schemas |

### Core Primitives

```
┌─────────────────────────────────────────────────────────────────┐
│                      LlamaIndex Workflow                        │
│                                                                 │
│   StartEvent ──► Step A ──► EventX ──► Step B ──► StopEvent   │
│                    │                    │                       │
│                    └──► EventY ──► Step C ──┘                  │
│                                                                 │
│   Context: shared key-value store across all steps             │
│   Event: typed Pydantic model -- the "message" between steps    │
│   Step: async function decorated with @step                    │
│   Workflow: orchestrator that routes events to steps           │
└─────────────────────────────────────────────────────────────────┘
```

**Key insight**: Steps are decoupled from each other. Step B doesn't call Step A directly -- it declares "I consume `EventX`", and the workflow routes accordingly. This makes workflows easy to test, extend, and parallelise.

## Project Overview: Automated BI Reporting Workflow

Our workflow automates a business intelligence report that normally requires an analyst to:

1. **Load & validate** raw sales data from multiple sources
2. **Retrieve context** -- pull relevant market benchmarks, product notes, and regional data from a knowledge base
3. **Compute KPIs** -- sales velocity, conversion rate, regional breakdown, product performance
4. **Detect anomalies** -- flag unusual patterns requiring attention
5. **Generate insights** -- produce structured analyst commentaries with evidence citations
6. **Format report** -- assemble findings into a validated executive report schema

### Workflow DAG

```
StartEvent
    │
    ├──► [ingest_data]  ──────────► DataReadyEvent
    │                                    │
    │              ┌──────────────────────┤
    │              ▼                      ▼
    │    [retrieve_context]      [compute_kpis]
    │              │                      │
    │        ContextReadyEvent      KPIsReadyEvent
    │              └──────────┬───────────┘
    │                         ▼
    │                [detect_anomalies]
    │                         │
    │                 AnomalyReportEvent
    │                         │
    │                [generate_insights]
    │                         │
    │                InsightsReadyEvent
    │                         │
    │                 [format_report]
    │                         │
    └─────────────────► StopEvent (ExecutiveReport)
```

## Learning Objectives

1. Build a **LlamaIndex-style Workflow** from scratch with typed events and steps
2. Implement a **shared Context object** that persists state across pipeline steps
3. Write **Pydantic-validated output models** for structured responses
4. Simulate **parallel step execution** for independent retrievals
5. Add **step-level retry and timeout** logic
6. Implement **tracing/observability** -- timing, input/output snapshots, step lineage
7. Understand when to use **Workflows vs simple chains vs agents**

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ["pandas", "numpy", "matplotlib", "seaborn", "plotly", "pydantic"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Environment ready.")

## Imports & Configuration

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import json, time, uuid, math, re, hashlib, textwrap
from datetime import datetime, timezone, timedelta
from dataclasses import dataclass, field, asdict
from typing import Any, Optional, Callable, Awaitable
from enum import Enum
from collections import defaultdict
import functools, asyncio, inspect, traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pydantic import BaseModel, Field, field_validator

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
np.random.seed(42)
print("Imports OK")

In [ ]:
# ── Simulated LLM (no API key needed) ────────────────────────────
class MockLLM:
    """Simulates an LLM for structured output generation.
    In production: use llama_index.llms.OpenAI, Ollama, etc.
    """
    def __init__(self, latency_ms: int = 50):
        self._latency = latency_ms / 1000

    def complete(self, prompt: str) -> str:
        time.sleep(self._latency)
        return f"[LLM response to: {prompt[:80]}...]"

    def structured_complete(self, prompt: str, schema: type) -> "BaseModel":
        """Return a populated schema instance based on the prompt context."""
        time.sleep(self._latency)
        return schema

llm = MockLLM(latency_ms=30)
print("MockLLM created (simulates OpenAI / Ollama calls)")

## Event System

LlamaIndex Workflows communicate via **typed events** -- Pydantic models that carry data between steps. This gives compile-time type safety, automatic validation, and clear contracts between steps.

### Why Typed Events?

- **Explicit contracts**: Step B declares it accepts `DataReadyEvent`, not "whatever Step A returns"
- **Validation at boundaries**: Pydantic validates event payloads before they reach the next step
- **Replayability**: Events can be serialized and replayed for debugging
- **Parallelism**: The event bus can fan-out one event to multiple listening steps simultaneously

In [ ]:
# ── Base event ────────────────────────────────────────────────────
class Event(BaseModel):
    """All workflow events inherit from this base."""
    event_id: str = Field(default_factory=lambda: str(uuid.uuid4())[:8])
    timestamp: str = Field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    metadata: dict = Field(default_factory=dict)

    model_config = {"arbitrary_types_allowed": True}


# ── Lifecycle events ──────────────────────────────────────────────────
class StartEvent(Event):
    """Begins workflow execution. Carries initial configuration."""
    config: dict = Field(default_factory=dict)


class StopEvent(Event):
    """Signals workflow completion. Carries the final result."""
    result: Any = None
    status: str = "success"


# ── Domain events ─────────────────────────────────────────────────────
class DataReadyEvent(Event):
    """Emitted after data ingestion and validation."""
    row_count: int
    column_count: int
    date_range: tuple
    source_id: str
    validation_passed: bool
    warnings: list[str] = Field(default_factory=list)


class ContextReadyEvent(Event):
    """Emitted after context retrieval from knowledge base."""
    retrieved_docs: list[dict]
    query_count: int
    top_k: int


class KPIsReadyEvent(Event):
    """Emitted after KPI computation."""
    kpis: dict
    period: str


class AnomalyReportEvent(Event):
    """Emitted after anomaly detection."""
    anomalies: list[dict]
    severity_counts: dict


class InsightsReadyEvent(Event):
    """Emitted after insight generation."""
    insights: list[dict]
    confidence_scores: list[float]


print("Event classes defined:", [
    c.__name__ for c in [StartEvent, StopEvent, DataReadyEvent,
                          ContextReadyEvent, KPIsReadyEvent,
                          AnomalyReportEvent, InsightsReadyEvent]
])

## Structured Output Models

Structured outputs are Pydantic models that define the **validated schema** for step results. LlamaIndex supports constraining LLM outputs to these schemas via `structured_predict()`. Here we show the same pattern with simulated data.

### Benefits of Structured Outputs

1. **No parsing logic**: The schema is the parser
2. **Guaranteed fields**: Downstream steps never get `KeyError`
3. **Type coercion**: Strings get converted to the right types automatically
4. **Validation hooks**: Custom validators enforce business rules at the model level

In [ ]:
# ── KPI schema ────────────────────────────────────────────────────
class SalesKPIs(BaseModel):
    """Structured KPI output produced by the compute_kpis step."""
    period: str
    total_revenue: float
    total_orders: int
    avg_order_value: float
    revenue_growth_pct: float
    order_growth_pct: float
    top_region: str
    top_product: str
    conversion_rate: float
    return_rate: float
    csat_score: float

    @field_validator("conversion_rate", "return_rate")
    @classmethod
    def pct_range(cls, v: float) -> float:
        if not (0.0 <= v <= 1.0):
            raise ValueError(f"Rate must be 0-1, got {v}")
        return v

    @field_validator("csat_score")
    @classmethod
    def csat_range(cls, v: float) -> float:
        if not (0.0 <= v <= 5.0):
            raise ValueError(f"CSAT must be 0-5, got {v}")
        return v


# ── Anomaly schema ────────────────────────────────────────────────────
class Anomaly(BaseModel):
    """A single detected anomaly."""
    anomaly_id: str = Field(default_factory=lambda: f"ANO-{uuid.uuid4().hex[:6].upper()}")
    field: str
    description: str
    severity: str   # "low" | "medium" | "high" | "critical"
    value: float
    expected_range: tuple
    evidence: str
    recommended_action: str

    @field_validator("severity")
    @classmethod
    def valid_severity(cls, v: str) -> str:
        if v not in {"low", "medium", "high", "critical"}:
            raise ValueError(f"Invalid severity: {v}")
        return v


# ── Insight schema ────────────────────────────────────────────────────
class AnalystInsight(BaseModel):
    """A single analyst-generated insight with citations."""
    insight_id: str = Field(default_factory=lambda: f"INS-{uuid.uuid4().hex[:6].upper()}")
    category: str       # "revenue" | "product" | "regional" | "risk" | "opportunity"
    headline: str
    body: str
    supporting_kpis: list[str]
    citations: list[str]
    confidence: float   # 0.0 - 1.0
    priority: int       # 1 = highest

    @field_validator("confidence")
    @classmethod
    def conf_range(cls, v: float) -> float:
        if not (0.0 <= v <= 1.0):
            raise ValueError(f"Confidence must be 0-1, got {v}")
        return v


# ── Executive report schema ───────────────────────────────────────────
class ExecutiveReport(BaseModel):
    """Final structured output -- the validated executive report."""
    report_id: str = Field(default_factory=lambda: f"RPT-{datetime.now().strftime('%Y%m%d-%H%M%S')}")
    generated_at: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    period: str
    executive_summary: str
    kpis: SalesKPIs
    anomalies: list[Anomaly]
    insights: list[AnalystInsight]
    data_sources: list[str]
    workflow_trace_id: str
    total_pipeline_ms: float

    @property
    def critical_anomaly_count(self) -> int:
        return sum(1 for a in self.anomalies if a.severity == "critical")

    @property
    def high_confidence_insights(self) -> list[AnalystInsight]:
        return [i for i in self.insights if i.confidence >= 0.8]


print("Structured output models defined:")
for m in [SalesKPIs, Anomaly, AnalystInsight, ExecutiveReport]:
    print(f"  {m.__name__}: {len(m.model_fields)} fields")

## Context Object

The `Context` is LlamaIndex Workflow's **shared memory** -- a typed key-value store accessible from every step. Steps read from and write to the context without needing to pass large dictionaries through event payloads.

### Context vs Event Payload

| | Event Payload | Context |
|---|---|---|
| **Scope** | Single step transition | Entire workflow run |
| **Access** | Receiving step only | Any step |
| **Best for** | Required inputs to next step | Accumulated state, shared data |
| **Size** | Keep small | Can be large |

### Thread Safety

LlamaIndex uses asyncio for step concurrency. `Context.get()` / `Context.set()` are atomic to prevent race conditions when parallel steps access the same keys.

In [ ]:
class Context:
    """Shared state store -- accessible from every step in the workflow.

    Simulates llama_index.core.workflow.Context.
    In production:
        async def my_step(self, ctx: Context, ev: SomeEvent):
            await ctx.set("my_key", value)
            prev = await ctx.get("my_key")
    """

    def __init__(self, workflow_id: str):
        self.workflow_id = workflow_id
        self._store: dict[str, Any] = {}
        self._metadata: dict[str, dict] = {}
        self._access_log: list[dict] = []

    def set(self, key: str, value: Any, step_name: str = ""):
        self._store[key] = value
        self._metadata[key] = {
            "set_by": step_name, "set_at": datetime.now(timezone.utc).isoformat(),
            "type": type(value).__name__,
        }
        self._access_log.append({"op": "set", "key": key, "step": step_name,
                                  "ts": time.time()})

    def get(self, key: str, default: Any = None, step_name: str = "") -> Any:
        self._access_log.append({"op": "get", "key": key, "step": step_name,
                                  "ts": time.time()})
        return self._store.get(key, default)

    def keys(self) -> list[str]:
        return list(self._store.keys())

    def summary(self) -> dict:
        return {
            "workflow_id": self.workflow_id,
            "keys": len(self._store),
            "access_log_entries": len(self._access_log),
            "key_list": list(self._store.keys()),
        }


print("Context class defined.")

## Tracing & Observability

Every step emits structured trace spans. This gives us a complete execution trace: which steps ran, how long each took, what they emitted, and whether any retried or failed.

In production LlamaIndex, this maps to the `instrumentation` layer with OpenTelemetry-compatible spans.

In [ ]:
@dataclass
class StepSpan:
    """Execution record for a single step."""
    span_id: str
    step_name: str
    workflow_id: str
    start_time: float
    end_time: float = 0.0
    status: str = "running"          # "running" | "ok" | "error" | "retried"
    input_event: str = ""
    output_event: str = ""
    retry_count: int = 0
    error_message: str = ""
    context_keys_written: list = field(default_factory=list)
    context_keys_read: list = field(default_factory=list)

    @property
    def duration_ms(self) -> float:
        return (self.end_time - self.start_time) * 1000 if self.end_time else 0.0

    def finish(self, output_event: str = "", status: str = "ok",
               error: str = ""):
        self.end_time = time.time()
        self.output_event = output_event
        self.status = status
        self.error_message = error


class WorkflowTracer:
    """Collects and reports step execution spans."""

    def __init__(self, workflow_id: str):
        self.workflow_id = workflow_id
        self.spans: list[StepSpan] = []
        self._active: dict[str, StepSpan] = {}

    def start_span(self, step_name: str, input_event: str) -> StepSpan:
        span = StepSpan(
            span_id=uuid.uuid4().hex[:8],
            step_name=step_name,
            workflow_id=self.workflow_id,
            start_time=time.time(),
            input_event=input_event,
        )
        self._active[step_name] = span
        self.spans.append(span)
        return span

    def finish_span(self, step_name: str, output_event: str = "",
                    status: str = "ok", error: str = ""):
        span = self._active.pop(step_name, None)
        if span:
            span.finish(output_event, status, error)

    def report(self) -> pd.DataFrame:
        if not self.spans:
            return pd.DataFrame()
        rows = []
        for s in self.spans:
            rows.append({
                "step": s.step_name, "status": s.status,
                "duration_ms": round(s.duration_ms, 2),
                "input": s.input_event, "output": s.output_event,
                "retries": s.retry_count,
            })
        return pd.DataFrame(rows)

    @property
    def total_ms(self) -> float:
        return sum(s.duration_ms for s in self.spans)


print("WorkflowTracer defined.")

## Knowledge Base (Context Retrieval Source)

Each step that needs external context calls `retrieve()` on a shared knowledge base. This simulates LlamaIndex's `VectorStoreIndex.as_retriever()` -- we use TF-IDF cosine similarity for the demo.

The knowledge base contains:
- Market benchmark reports
- Regional economic indicators
- Product category benchmarks
- Industry conversion rate averages

In [ ]:
KB_DOCUMENTS = [
    {"doc_id": "mkt-001", "title": "E-Commerce Market Benchmarks 2026",
     "text": (
         "Global e-commerce revenue grew 14.3% YoY in Q1 2026. Average order value "
         "across B2C segments is $94.50. Top-performing categories include Electronics "
         "(22% of revenue), Apparel (18%), and Home & Garden (15%). Cart abandonment "
         "rates average 68% industry-wide. Mobile commerce accounts for 57% of all "
         "transactions. Conversion rates: Electronics 2.1%, Apparel 3.4%, Home 2.8%."
     )},
    {"doc_id": "reg-001", "title": "North America Regional Performance Report",
     "text": (
         "North America region shows strongest performance with avg revenue per user "
         "of $210/quarter. Seasonal peaks: November (+35%), December (+28%), "
         "February (-12%). Urban vs rural split: 72% urban, 28% rural. Top states "
         "by revenue: California (18%), Texas (12%), New York (11%). Return rates "
         "in NA average 8.2% -- below global average of 9.1%."
     )},
    {"doc_id": "reg-002", "title": "EMEA Regional Performance Report",
     "text": (
         "EMEA region growing at 11.2% annually. Germany leads at 23% of EMEA "
         "revenue, followed by UK (19%) and France (14%). Average order value €87. "
         "VAT compliance requirements added 4-6% operational overhead. "
         "Localisation (language/currency) increases conversion by 23%. "
         "Return rates higher at 12.4% driven by German consumer protection laws."
     )},
    {"doc_id": "reg-003", "title": "APAC Sales Intelligence Report",
     "text": (
         "APAC shows fastest growth at 18.7% YoY. Japan and South Korea lead in "
         "premium segment (AOV $145). Southeast Asia shows high volume, lower AOV "
         "at $42. Mobile-first penetration is 79% across APAC. Social commerce "
         "accounts for 31% of APAC orders. Payment method preference: digital "
         "wallets (46%), BNPL (22%), credit card (21%)."
     )},
    {"doc_id": "prod-001", "title": "Electronics Category Benchmark",
     "text": (
         "Electronics category benchmarks: AOV $185-$210, conversion rate 1.8-2.4%, "
         "return rate 11-14% (higher due to compatibility issues). "
         "Warranty claims average 3.2% of revenue. Bundle upsell rate 28%. "
         "Search-to-purchase path averages 4.3 sessions. Refurbished items growing "
         "at 22% YoY, margin premium 8% vs new."
     )},
    {"doc_id": "prod-002", "title": "Apparel & Fashion Category Benchmark",
     "text": (
         "Apparel benchmarks: AOV $65-$85, conversion rate 3.0-3.8%, "
         "return rate 18-24% (size/fit issues). Seasonal markdown depth averages 35%. "
         "Loyalty program members generate 2.4x more revenue per visit. "
         "Size inclusivity (range 2XS-4XL) correlates with +15% conversion. "
         "Personalised recommendations drive 31% of apparel revenue."
     )},
    {"doc_id": "risk-001", "title": "E-Commerce Risk & Fraud Intelligence",
     "text": (
         "Fraud rates industry average: 0.6% of transaction value. High-risk signals: "
         "order values >3× AOV, new accounts with high-value orders, rapid sequential "
         "orders, mismatched billing/shipping regions. Chargeback rates above 1% "
         "trigger payment processor scrutiny. Account takeover attacks up 34% in 2026. "
         "BNPL fraud growing at 28% -- higher due to weaker identity checks."
     )},
    {"doc_id": "ops-001", "title": "Supply Chain & Fulfilment Benchmarks",
     "text": (
         "Same-day delivery adoption at 23% of orders in major metro areas. "
         "Expedited shipping (2-day) requested by 41% of customers. "
         "Out-of-stock rates above 3% result in 11% cart abandonment increase. "
         "Average fulfilment cost $6.80 per order for standard shipping. "
         "Returns processing cost $4.20 per item. Cross-border orders add $12-$18 "
         "in fulfilment overhead due to customs and compliance."
     )},
    {"doc_id": "cust-001", "title": "Customer Experience & CSAT Benchmarks",
     "text": (
         "Industry average CSAT for e-commerce: 3.8/5. NPS median: +31. "
         "Top CSAT drivers: delivery speed (32%), product accuracy (28%), "
         "easy returns (21%), customer support responsiveness (19%). "
         "CSAT below 3.5 correlates with 40% higher churn in 12 months. "
         "Personalisation lifts CSAT by 0.3 points on average. "
         "Live chat support improves CSAT 0.4 points vs email-only support."
     )},
    {"doc_id": "fin-001", "title": "Financial Metrics & Margin Benchmarks",
     "text": (
         "E-commerce gross margin benchmarks: Electronics 18-22%, Apparel 45-55%, "
         "Home 38-44%, Beauty 58-65%. EBITDA margins for scaled operators: 8-12%. "
         "Marketing as % of revenue: 12-18% for growth stage, 8-12% for mature. "
         "Customer acquisition cost (CAC) industry median: $38. "
         "Customer lifetime value (LTV) to CAC ratio benchmark: 3:1 minimum. "
         "Subscription/recurring revenue premium in valuation: 2-3× revenue multiple."
     )},
]

print(f"Knowledge base: {len(KB_DOCUMENTS)} documents")
for d in KB_DOCUMENTS:
    print(f"  [{d['doc_id']}] {d['title']}")

In [ ]:
import re as _re, math as _math
from collections import Counter as _Counter

class KnowledgeBase:
    """Simulates a LlamaIndex VectorStoreIndex retriever using TF-IDF."""

    def __init__(self, documents: list[dict]):
        self.documents = documents
        self._build_index()

    def _tokenize(self, text: str) -> list[str]:
        return _re.findall(r"[a-z0-9]+", text.lower())

    def _build_index(self):
        self._df: dict[str, int] = _Counter()
        self._doc_tokens: list[list[str]] = []
        for doc in self.documents:
            tokens = self._tokenize(doc["text"])
            self._doc_tokens.append(tokens)
            for t in set(tokens):
                self._df[t] += 1

        n = len(self.documents)
        self._idf = {t: _math.log(n / (1 + df)) for t, df in self._df.items()}

        self._doc_vecs: list[dict] = []
        for tokens in self._doc_tokens:
            tf = _Counter(tokens)
            vec = {t: tf[t] * self._idf.get(t, 0) for t in tf}
            norm = _math.sqrt(sum(v**2 for v in vec.values())) or 1.0
            self._doc_vecs.append({t: v / norm for t, v in vec.items()})

    def _score(self, query_vec: dict, doc_vec: dict) -> float:
        return sum(query_vec.get(t, 0) * doc_vec.get(t, 0)
                   for t in query_vec)

    def retrieve(self, query: str, top_k: int = 3) -> list[dict]:
        tokens = self._tokenize(query)
        tf = _Counter(tokens)
        qvec = {t: tf[t] * self._idf.get(t, 0) for t in tf}
        qnorm = _math.sqrt(sum(v**2 for v in qvec.values())) or 1.0
        qvec = {t: v / qnorm for t, v in qvec.items()}

        scored = [(self._score(qvec, dv), doc)
                  for dv, doc in zip(self._doc_vecs, self.documents)]
        scored.sort(key=lambda x: -x[0])
        return [
            {"score": round(s, 4), "doc_id": d["doc_id"],
             "title": d["title"], "snippet": d["text"][:200]}
            for s, d in scored[:top_k] if s > 0
        ]


kb = KnowledgeBase(KB_DOCUMENTS)

# Quick test
sample = kb.retrieve("conversion rate electronics benchmark", top_k=3)
print("Retrieval test (electronics benchmarks):")
for r in sample:
    print(f"  [{r['doc_id']}] score={r['score']:.4f} -- {r['title']}")

## Simulated Sales Data

We generate 90 days of realistic multi-region sales data to drive the workflow.

In [ ]:
def generate_sales_data(n_days: int = 90, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    regions = ["North America", "EMEA", "APAC", "LATAM"]
    products = ["Electronics", "Apparel", "Home & Garden", "Beauty", "Sports"]
    channels = ["organic_search", "paid_search", "social", "email", "direct"]

    region_weights = [0.40, 0.30, 0.22, 0.08]
    product_weights = [0.30, 0.25, 0.20, 0.15, 0.10]
    aov_by_product = {"Electronics": 185, "Apparel": 72, "Home & Garden": 95,
                      "Beauty": 48, "Sports": 62}
    base_orders_per_day = 420

    rows = []
    start = datetime(2026, 1, 12)
    for day in range(n_days):
        date = start + timedelta(days=day)
        weekday_factor = 0.85 if date.weekday() >= 5 else 1.0
        trend_factor = 1 + 0.003 * day

        n_orders = int(rng.poisson(base_orders_per_day * weekday_factor * trend_factor))

        for _ in range(n_orders):
            region = rng.choice(regions, p=region_weights)
            product = rng.choice(products, p=product_weights)
            channel = rng.choice(channels)
            base_aov = aov_by_product[product]
            order_value = max(5.0, rng.normal(base_aov, base_aov * 0.25))
            is_return = rng.random() < 0.09
            rating = rng.choice([1, 2, 3, 4, 5], p=[0.04, 0.07, 0.14, 0.35, 0.40])

            rows.append({
                "date": date.date(),
                "region": region,
                "product_category": product,
                "channel": channel,
                "order_value": round(order_value, 2),
                "units": rng.integers(1, 5),
                "is_return": is_return,
                "csat_rating": rating,
                "order_id": uuid.uuid4().hex[:12],
            })

    return pd.DataFrame(rows)


RAW_DATA = generate_sales_data(n_days=90)
print(f"Generated sales data: {len(RAW_DATA):,} orders over 90 days")
print(f"Columns: {list(RAW_DATA.columns)}")
print(f"Date range: {RAW_DATA['date'].min()} → {RAW_DATA['date'].max()}")
print(f"Total revenue: ${RAW_DATA['order_value'].sum():,.2f}")

## Workflow Engine

The `Workflow` class is the orchestrator -- it routes events to steps, manages the context, collects traces, and handles retries.

### How Routing Works

```python
# Each step declares what event(s) it accepts:
@workflow.step
def compute_kpis(ctx: Context, ev: DataReadyEvent) -> KPIsReadyEvent:
    ...

# When DataReadyEvent is emitted, the engine finds all steps that accept it
# and calls them (possibly in parallel for fan-out patterns).
```

### Retry Policy

Steps can declare a retry policy. The engine transparently retries on exception with exponential backoff before propagating the error.

In [ ]:
@dataclass
class RetryPolicy:
    max_retries: int = 2
    base_delay_s: float = 0.1
    exponential_backoff: bool = True

    def delay(self, attempt: int) -> float:
        if self.exponential_backoff:
            return self.base_delay_s * (2 ** attempt)
        return self.base_delay_s


class Workflow:
    """Orchestrates typed-event-driven step execution.

    Simulates llama_index.core.workflow.Workflow.

    In production:
        class MyWorkflow(Workflow):
            @step
            async def my_step(self, ctx: Context, ev: StartEvent) -> StopEvent:
                data = await ctx.get("data")
                return StopEvent(result=data)
    """

    def __init__(self, name: str = "workflow",
                 default_retry: RetryPolicy = None):
        self.name = name
        self._steps: dict[str, dict] = {}    # event_type -> step meta
        self._default_retry = default_retry or RetryPolicy()
        self.workflow_id = uuid.uuid4().hex[:12]
        self.tracer = WorkflowTracer(self.workflow_id)

    def step(self, fn: Callable = None,
             retry: RetryPolicy = None,
             timeout_s: float = None):
        """Decorator that registers a function as a workflow step."""
        def decorator(func: Callable):
            hints = func.__annotations__
            input_type = None
            output_type = hints.get("return")
            for name, hint in hints.items():
                if name == "return":
                    continue
                if isinstance(hint, type) and issubclass(hint, Event):
                    input_type = hint
                    break

            self._steps[input_type.__name__ if input_type else func.__name__] = {
                "fn": func, "input_type": input_type,
                "output_type": output_type,
                "retry": retry or self._default_retry,
                "timeout_s": timeout_s,
                "name": func.__name__,
            }
            return func

        return decorator(fn) if fn else decorator

    def _find_steps(self, event: Event) -> list[dict]:
        event_type = type(event).__name__
        return [s for s in self._steps.values()
                if s["input_type"] and s["input_type"].__name__ == event_type]

    def _run_step(self, step_meta: dict, ctx: Context, event: Event) -> Event:
        fn = step_meta["fn"]
        retry: RetryPolicy = step_meta["retry"]
        span = self.tracer.start_span(step_meta["name"], type(event).__name__)

        for attempt in range(retry.max_retries + 1):
            try:
                result = fn(ctx, event)
                span.finish(output_event=type(result).__name__ if result else "",
                            status="ok")
                return result
            except Exception as exc:
                if attempt < retry.max_retries:
                    span.retry_count += 1
                    time.sleep(retry.delay(attempt))
                else:
                    span.finish(status="error",
                                error=f"{type(exc).__name__}: {exc}")
                    raise

    def run(self, start_event: StartEvent, ctx: Context = None) -> StopEvent:
        if ctx is None:
            ctx = Context(workflow_id=self.workflow_id)

        ctx.set("workflow_id", self.workflow_id, "engine")
        ctx.set("start_time", time.time(), "engine")

        current_events = [start_event]
        final_event = None

        while current_events:
            next_events = []
            for event in current_events:
                if isinstance(event, StopEvent):
                    final_event = event
                    continue
                steps = self._find_steps(event)
                if not steps:
                    continue
                for step_meta in steps:
                    result = self._run_step(step_meta, ctx, event)
                    if result is not None:
                        next_events.append(result)
            current_events = next_events

        return final_event or StopEvent(result=None, status="no_stop_event")


print("Workflow engine defined.")

## Workflow Steps

Now we implement each step in the BI reporting workflow. Each step:
1. Reads needed state from `Context`
2. Does its work (data processing, retrieval, or generation)
3. Writes outputs back to `Context`
4. Returns a typed `Event` to trigger the next step(s)

### Step 1: `ingest_data` -- Load & Validate

Accepts `StartEvent`. Loads the raw data, validates schema and quality, then emits `DataReadyEvent`.

In [ ]:
def step_ingest_data(ctx: Context, ev: StartEvent) -> DataReadyEvent:
    config = ev.config
    data_source = config.get("data_source", "builtin")

    # Load data (in production: read from S3, database, API, etc.)
    df = RAW_DATA.copy()

    # ── Validation ────────────────────────────────────────────────────
    warnings_list = []

    # Required columns
    required = ["date", "region", "product_category", "order_value",
                "is_return", "csat_rating", "channel"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Null check
    null_counts = df[required].isnull().sum()
    if null_counts.any():
        warnings_list.append(f"Nulls found: {null_counts[null_counts > 0].to_dict()}")

    # Order value range
    neg_orders = (df["order_value"] <= 0).sum()
    if neg_orders > 0:
        warnings_list.append(f"{neg_orders} orders with non-positive value")

    # Date range sanity
    date_range = (str(df["date"].min()), str(df["date"].max()))

    # Write validated data to context
    ctx.set("raw_data", df, "ingest_data")
    ctx.set("data_source", data_source, "ingest_data")
    ctx.set("date_range", date_range, "ingest_data")

    print(f"  [ingest_data] rows={len(df):,} cols={len(df.columns)} "
          f"warnings={len(warnings_list)}")

    return DataReadyEvent(
        row_count=len(df), column_count=len(df.columns),
        date_range=date_range, source_id=data_source,
        validation_passed=len(missing) == 0,
        warnings=warnings_list,
    )

print("step_ingest_data defined.")

### Step 2a: `retrieve_context` -- Knowledge Base Lookup

Accepts `DataReadyEvent`. Generates retrieval queries from the data profile, fetches relevant benchmark docs from the knowledge base, and emits `ContextReadyEvent`.

This step runs **in parallel** with `compute_kpis` -- both accept `DataReadyEvent`.

In [ ]:
def step_retrieve_context(ctx: Context, ev: DataReadyEvent) -> ContextReadyEvent:
    df: pd.DataFrame = ctx.get("raw_data", step_name="retrieve_context")

    # Build context-aware retrieval queries from the data
    top_product = (df.groupby("product_category")["order_value"]
                     .sum().idxmax())
    top_region = (df.groupby("region")["order_value"]
                    .sum().idxmax())
    avg_csat = df["csat_rating"].mean()
    return_rate = df["is_return"].mean()

    queries = [
        f"{top_product} category benchmark conversion rate AOV",
        f"{top_region} regional performance e-commerce",
        "e-commerce fraud risk high order value anomaly",
        "customer satisfaction CSAT benchmark e-commerce",
        f"return rate {'high' if return_rate > 0.12 else 'normal'} "
        f"ecommerce {top_product}",
        "supply chain fulfilment benchmark operational cost",
        "financial margin gross profit e-commerce benchmark",
    ]

    retrieved_docs = []
    for query in queries:
        results = kb.retrieve(query, top_k=2)
        retrieved_docs.extend(results)

    # Deduplicate by doc_id, keep best score
    seen: dict[str, dict] = {}
    for doc in retrieved_docs:
        if doc["doc_id"] not in seen or doc["score"] > seen[doc["doc_id"]]["score"]:
            seen[doc["doc_id"]] = doc
    unique_docs = sorted(seen.values(), key=lambda x: -x["score"])

    # Store in context for later steps
    ctx.set("retrieved_context", unique_docs, "retrieve_context")
    ctx.set("retrieval_queries", queries, "retrieve_context")

    print(f"  [retrieve_context] queries={len(queries)} "
          f"unique_docs={len(unique_docs)}")

    return ContextReadyEvent(
        retrieved_docs=unique_docs,
        query_count=len(queries),
        top_k=2,
    )

print("step_retrieve_context defined.")

### Step 2b: `compute_kpis` -- Sales Analytics

Also accepts `DataReadyEvent` (parallel with `retrieve_context`). Computes all KPIs and writes a `SalesKPIs` structured output to context.

In [ ]:
def step_compute_kpis(ctx: Context, ev: DataReadyEvent) -> KPIsReadyEvent:
    df: pd.DataFrame = ctx.get("raw_data", step_name="compute_kpis")
    date_range = ctx.get("date_range", step_name="compute_kpis")

    # Period label
    period = f"{date_range[0]} to {date_range[1]}"

    # Current vs prior period split (first 45 days = prior, last 45 = current)
    mid = df["date"].median()
    prior = df[df["date"] <= mid]
    current = df[df["date"] > mid]

    def safe_growth(curr, prev):
        if prev == 0:
            return 0.0
        return round((curr - prev) / prev * 100, 2)

    revenue_curr = current["order_value"].sum()
    revenue_prev = prior["order_value"].sum()
    orders_curr = len(current)
    orders_prev = len(prior)

    top_region = (df.groupby("region")["order_value"]
                    .sum().idxmax())
    top_product = (df.groupby("product_category")["order_value"]
                     .sum().idxmax())

    kpis = SalesKPIs(
        period=period,
        total_revenue=round(df["order_value"].sum(), 2),
        total_orders=len(df),
        avg_order_value=round(df["order_value"].mean(), 2),
        revenue_growth_pct=safe_growth(revenue_curr, revenue_prev),
        order_growth_pct=safe_growth(orders_curr, orders_prev),
        top_region=top_region,
        top_product=top_product,
        conversion_rate=round(min(0.99, max(0.0, np.random.normal(0.028, 0.003))), 4),
        return_rate=round(df["is_return"].mean(), 4),
        csat_score=round(df["csat_rating"].mean(), 2),
    )

    # Regional and product breakdowns
    region_rev = (df.groupby("region")["order_value"]
                    .agg(revenue="sum", orders="count")
                    .round(2).reset_index()
                    .sort_values("revenue", ascending=False))
    product_rev = (df.groupby("product_category")["order_value"]
                     .agg(revenue="sum", orders="count")
                     .round(2).reset_index()
                     .sort_values("revenue", ascending=False))
    channel_rev = (df.groupby("channel")["order_value"]
                     .agg(revenue="sum", orders="count")
                     .round(2).reset_index()
                     .sort_values("revenue", ascending=False))

    daily_rev = (df.groupby("date")["order_value"]
                   .sum().reset_index()
                   .rename(columns={"order_value": "revenue"}))

    ctx.set("kpis", kpis, "compute_kpis")
    ctx.set("region_breakdown", region_rev, "compute_kpis")
    ctx.set("product_breakdown", product_rev, "compute_kpis")
    ctx.set("channel_breakdown", channel_rev, "compute_kpis")
    ctx.set("daily_revenue", daily_rev, "compute_kpis")
    ctx.set("prior_df", prior, "compute_kpis")
    ctx.set("current_df", current, "compute_kpis")

    print(f"  [compute_kpis] revenue=${kpis.total_revenue:,.0f} "
          f"orders={kpis.total_orders:,} "
          f"growth={kpis.revenue_growth_pct:+.1f}%")

    return KPIsReadyEvent(kpis=kpis.model_dump(), period=period)

print("step_compute_kpis defined.")

### Step 3: `detect_anomalies` -- Statistical Outlier Detection

This step waits for **both** `ContextReadyEvent` AND `KPIsReadyEvent` before running. In LlamaIndex Workflows, this is a **join pattern** -- the step waits for multiple events and fires only when all are available.

We detect anomalies by comparing KPIs against industry benchmarks retrieved in the previous step.

In [ ]:
# ── Join gate: waits for both ContextReadyEvent and KPIsReadyEvent ──
# In production LlamaIndex this is:
#   @step
#   async def detect_anomalies(self, ctx: Context,
#                              ctx_ev: ContextReadyEvent,
#                              kpi_ev: KPIsReadyEvent) -> AnomalyReportEvent:
#
# Here we use a simple merge adapter since we don't have the full async engine:

def step_detect_anomalies(ctx: Context, ev: KPIsReadyEvent) -> AnomalyReportEvent:
    """Detect anomalies using KPIs and retrieved benchmarks."""
    kpis: SalesKPIs = ctx.get("kpis", step_name="detect_anomalies")
    context_docs = ctx.get("retrieved_context", default=[], step_name="detect_anomalies")
    df: pd.DataFrame = ctx.get("raw_data", step_name="detect_anomalies")

    anomalies: list[Anomaly] = []

    # ── 1. Return rate vs benchmark ───────────────────────────────────
    benchmark_return = 0.091   # from reg-001 doc
    if kpis.return_rate > benchmark_return * 1.3:
        anomalies.append(Anomaly(
            field="return_rate",
            description=(f"Return rate {kpis.return_rate:.1%} exceeds industry "
                         f"benchmark of {benchmark_return:.1%} by more than 30%"),
            severity="high" if kpis.return_rate > 0.15 else "medium",
            value=kpis.return_rate,
            expected_range=(0.05, benchmark_return),
            evidence="Source: E-Commerce Risk Intelligence (risk-001)",
            recommended_action=(
                "Investigate size/fit accuracy for Apparel; "
                "review product descriptions for Electronics"
            ),
        ))

    # ── 2. CSAT vs benchmark ──────────────────────────────────────────
    benchmark_csat = 3.8
    if kpis.csat_score < benchmark_csat - 0.3:
        anomalies.append(Anomaly(
            field="csat_score",
            description=(f"CSAT {kpis.csat_score:.2f} is below industry benchmark "
                         f"{benchmark_csat}"),
            severity="high" if kpis.csat_score < 3.2 else "medium",
            value=kpis.csat_score,
            expected_range=(3.8, 5.0),
            evidence="Source: Customer Experience Benchmarks (cust-001)",
            recommended_action="Schedule customer feedback analysis; review NPS drivers",
        ))

    # ── 3. Revenue growth outlier ─────────────────────────────────────
    daily_rev: pd.DataFrame = ctx.get("daily_revenue", step_name="detect_anomalies")
    if daily_rev is not None and len(daily_rev) > 14:
        rolling_std = daily_rev["revenue"].rolling(7).std().dropna()
        rolling_mean = daily_rev["revenue"].rolling(7).mean().dropna()
        last_rev = daily_rev["revenue"].iloc[-1]
        last_mean = rolling_mean.iloc[-1]
        last_std = rolling_std.iloc[-1]
        if last_std > 0 and abs(last_rev - last_mean) > 2.5 * last_std:
            direction = "spike" if last_rev > last_mean else "drop"
            anomalies.append(Anomaly(
                field="daily_revenue",
                description=(f"Day-over-day revenue {direction}: "
                             f"${last_rev:,.0f} vs 7-day avg ${last_mean:,.0f}"),
                severity="high" if abs(last_rev - last_mean) > 3 * last_std
                          else "medium",
                value=round(last_rev, 2),
                expected_range=(round(last_mean - 2*last_std, 2),
                                round(last_mean + 2*last_std, 2)),
                evidence="7-day rolling mean + 2.5σ threshold",
                recommended_action=(
                    "Check marketing campaign timing; "
                    "verify data pipeline integrity"
                ),
            ))

    # ── 4. Conversion rate vs benchmark ──────────────────────────────
    benchmark_cr = 0.028
    if abs(kpis.conversion_rate - benchmark_cr) > benchmark_cr * 0.35:
        direction = "above" if kpis.conversion_rate > benchmark_cr else "below"
        anomalies.append(Anomaly(
            field="conversion_rate",
            description=(f"Conversion rate {kpis.conversion_rate:.2%} "
                         f"is {direction} benchmark {benchmark_cr:.2%}"),
            severity="medium",
            value=kpis.conversion_rate,
            expected_range=(0.018, 0.038),
            evidence="Source: E-Commerce Market Benchmarks 2026 (mkt-001)",
            recommended_action=(
                "A/B test checkout flow variants; "
                "audit funnel drop-off points"
            ),
        ))

    # ── 5. Channel concentration risk ────────────────────────────────
    channel_share = (df.groupby("channel")["order_value"].sum() /
                     df["order_value"].sum())
    max_channel_share = channel_share.max()
    if max_channel_share > 0.45:
        top_channel = channel_share.idxmax()
        anomalies.append(Anomaly(
            field="channel_concentration",
            description=(f"Channel '{top_channel}' contributes "
                         f"{max_channel_share:.0%} of revenue -- "
                         f"high dependency risk"),
            severity="medium",
            value=round(max_channel_share, 4),
            expected_range=(0.0, 0.40),
            evidence="Diversification benchmark: no single channel >40% of revenue",
            recommended_action=(
                "Invest in channel diversification; "
                "reduce single-channel dependency risk"
            ),
        ))

    ctx.set("anomalies", anomalies, "detect_anomalies")

    severity_counts = {"critical": 0, "high": 0, "medium": 0, "low": 0}
    for a in anomalies:
        severity_counts[a.severity] += 1

    print(f"  [detect_anomalies] found={len(anomalies)} "
          f"high={severity_counts['high']} medium={severity_counts['medium']}")

    return AnomalyReportEvent(
        anomalies=[a.model_dump() for a in anomalies],
        severity_counts=severity_counts,
    )

print("step_detect_anomalies defined.")

### Step 4: `generate_insights` -- Synthesise Analyst Commentary

Reads KPIs, anomalies, and retrieved context to produce `AnalystInsight` structured outputs. In production this step calls an LLM with structured output constraints. Here we simulate the logic with rule-based synthesis that mirrors what an LLM would produce.

In [ ]:
def step_generate_insights(ctx: Context,
                            ev: AnomalyReportEvent) -> InsightsReadyEvent:
    kpis: SalesKPIs = ctx.get("kpis", step_name="generate_insights")
    anomalies: list[Anomaly] = ctx.get("anomalies", default=[], step_name="generate_insights")
    context_docs = ctx.get("retrieved_context", default=[], step_name="generate_insights")
    region_df: pd.DataFrame = ctx.get("region_breakdown", step_name="generate_insights")
    product_df: pd.DataFrame = ctx.get("product_breakdown", step_name="generate_insights")

    # Build citation lookup from retrieved docs
    citations = {d["doc_id"]: d["title"] for d in context_docs}

    insights: list[AnalystInsight] = []

    # ── Insight 1: Revenue growth narrative ──────────────────────────
    growth_tone = "strong" if kpis.revenue_growth_pct > 5 else (
        "modest" if kpis.revenue_growth_pct > 0 else "declining")
    insights.append(AnalystInsight(
        category="revenue",
        headline=(f"{growth_tone.capitalize()} revenue growth of "
                  f"{kpis.revenue_growth_pct:+.1f}% period-over-period"),
        body=(
            f"Total revenue of ${kpis.total_revenue:,.0f} across "
            f"{kpis.total_orders:,} orders ({kpis.period}). "
            f"Average order value of ${kpis.avg_order_value:.2f} "
            f"{'is above' if kpis.avg_order_value > 94.5 else 'is below'} the "
            f"industry benchmark of $94.50 (mkt-001). "
            f"Revenue growth of {kpis.revenue_growth_pct:+.1f}% compares "
            f"{'favourably' if kpis.revenue_growth_pct > 3 else 'unfavourably'} "
            f"to the sector average of +14.3% YoY (mkt-001)."
        ),
        supporting_kpis=["total_revenue", "avg_order_value", "revenue_growth_pct"],
        citations=["mkt-001", "fin-001"],
        confidence=0.92,
        priority=1,
    ))

    # ── Insight 2: Regional performance ──────────────────────────────
    top_rev = region_df.iloc[0]
    bottom_rev = region_df.iloc[-1]
    region_gap = top_rev["revenue"] / bottom_rev["revenue"]
    insights.append(AnalystInsight(
        category="regional",
        headline=(f"{top_rev['region']} leads revenue; "
                  f"{region_gap:.1f}× gap to {bottom_rev['region']}"),
        body=(
            f"{top_rev['region']} generated ${top_rev['revenue']:,.0f} "
            f"({top_rev['orders']:,} orders), consistent with its benchmark "
            f"dominance in AOV and purchase frequency (reg-001). "
            f"{bottom_rev['region']} at ${bottom_rev['revenue']:,.0f} shows "
            f"growth potential -- APAC mobile-first trends suggest digital "
            f"channel investment could accelerate performance (reg-003)."
        ),
        supporting_kpis=["top_region"],
        citations=list(set(["reg-001", "reg-002", "reg-003"]) &
                       set(citations.keys())) or ["reg-001"],
        confidence=0.88,
        priority=2,
    ))

    # ── Insight 3: Product mix ────────────────────────────────────────
    top_prod = product_df.iloc[0]
    insights.append(AnalystInsight(
        category="product",
        headline=(f"{top_prod['product_category']} drives "
                  f"{top_prod['revenue']/kpis.total_revenue:.0%} of total revenue"),
        body=(
            f"{top_prod['product_category']} leads with "
            f"${top_prod['revenue']:,.0f} across {top_prod['orders']:,} orders. "
            f"Industry benchmark shows "
            f"{'Electronics is the top category at 22% of revenue' if top_prod['product_category'] == 'Electronics' else top_prod['product_category'] + ' growth outpacing Electronics benchmark'}. "
            f"Margin profile for {top_prod['product_category']} suggests "
            f"{'18-22% gross margin -- tightest in portfolio' if top_prod['product_category'] == 'Electronics' else '45-55% gross margin opportunity'} (fin-001)."
        ),
        supporting_kpis=["top_product"],
        citations=["prod-001" if top_prod["product_category"] == "Electronics"
                   else "prod-002", "fin-001"],
        confidence=0.85,
        priority=3,
    ))

    # ── Insight 4: Customer experience ───────────────────────────────
    csat_vs_bench = kpis.csat_score - 3.8
    insights.append(AnalystInsight(
        category="risk" if csat_vs_bench < -0.2 else "opportunity",
        headline=(f"CSAT {kpis.csat_score:.2f}/5 is "
                  f"{'below' if csat_vs_bench < 0 else 'above'} "
                  f"benchmark by {abs(csat_vs_bench):.2f} points"),
        body=(
            f"Customer satisfaction score of {kpis.csat_score:.2f} "
            f"{'requires attention -- CSAT below 3.5 correlates with 40% higher churn' if kpis.csat_score < 3.5 else 'is healthy'}. "
            f"Return rate of {kpis.return_rate:.1%} "
            f"{'exceeds' if kpis.return_rate > 0.091 else 'is within'} "
            f"the NA benchmark of 8.2% (reg-001). "
            f"Personalisation investment projected to lift CSAT by +0.3 points (cust-001)."
        ),
        supporting_kpis=["csat_score", "return_rate"],
        citations=["cust-001", "reg-001"],
        confidence=0.80,
        priority=4,
    ))

    # ── Insight 5: Anomaly-driven risk ────────────────────────────────
    high_anomalies = [a for a in anomalies if a.severity in ("high", "critical")]
    if high_anomalies:
        fields = ", ".join(a.field for a in high_anomalies)
        insights.append(AnalystInsight(
            category="risk",
            headline=f"Elevated risk signals detected: {fields}",
            body=(
                f"{len(high_anomalies)} high-severity anomal"
                f"{'y' if len(high_anomalies) == 1 else 'ies'} flagged. "
                + " ".join(
                    f"[{a.field}] {a.description}; action: {a.recommended_action}."
                    for a in high_anomalies[:2]
                )
            ),
            supporting_kpis=[a.field for a in high_anomalies],
            citations=["risk-001"],
            confidence=0.95,
            priority=1,
        ))

    ctx.set("insights", insights, "generate_insights")

    print(f"  [generate_insights] produced={len(insights)} insights "
          f"avg_confidence={np.mean([i.confidence for i in insights]):.2f}")

    return InsightsReadyEvent(
        insights=[i.model_dump() for i in insights],
        confidence_scores=[i.confidence for i in insights],
    )

print("step_generate_insights defined.")

### Step 5: `format_report` -- Assemble & Validate Executive Report

Final step -- collects all context, builds the `ExecutiveReport` Pydantic model, validates it, and emits `StopEvent`.

In [ ]:
def step_format_report(ctx: Context, ev: InsightsReadyEvent) -> StopEvent:
    kpis: SalesKPIs = ctx.get("kpis", step_name="format_report")
    anomalies: list[Anomaly] = ctx.get("anomalies", default=[], step_name="format_report")
    insights: list[AnalystInsight] = ctx.get("insights", default=[], step_name="format_report")
    context_docs = ctx.get("retrieved_context", default=[], step_name="format_report")
    workflow_id = ctx.get("workflow_id", step_name="format_report")
    start_time = ctx.get("start_time", step_name="format_report")
    date_range = ctx.get("date_range", step_name="format_report")

    total_ms = (time.time() - start_time) * 1000 if start_time else 0.0

    # Executive summary (rule-based, would be LLM in production)
    growth_word = ("grew" if kpis.revenue_growth_pct > 0 else "declined")
    exec_summary = (
        f"Sales performance for {kpis.period}: total revenue "
        f"${kpis.total_revenue:,.0f} across {kpis.total_orders:,} orders. "
        f"Revenue {growth_word} {abs(kpis.revenue_growth_pct):.1f}% vs prior period. "
        f"{kpis.top_region} led by region; {kpis.top_product} by category. "
        f"CSAT {kpis.csat_score:.2f}/5 "
        f"({'above' if kpis.csat_score >= 3.8 else 'below'} industry benchmark). "
        f"{len([a for a in anomalies if a.severity in ('high','critical')])} "
        f"high-priority anomalies require attention. "
        f"{len(insights)} analyst insights generated with "
        f"{len([i for i in insights if i.confidence >= 0.85])}"
        f" high-confidence findings."
    )

    report = ExecutiveReport(
        period=kpis.period,
        executive_summary=exec_summary,
        kpis=kpis,
        anomalies=anomalies,
        insights=sorted(insights, key=lambda x: x.priority),
        data_sources=list({d["doc_id"] for d in context_docs}),
        workflow_trace_id=workflow_id or "unknown",
        total_pipeline_ms=round(total_ms, 2),
    )

    ctx.set("report", report, "format_report")

    print(f"  [format_report] report_id={report.report_id} "
          f"pipeline={report.total_pipeline_ms:.0f}ms")

    return StopEvent(result=report, status="success")

print("step_format_report defined.")

## Register Steps & Run the Workflow

Now we wire everything together -- register each step with its event handler, then call `run()`.

In [ ]:
# ── Build workflow ────────────────────────────────────────────────
bi_workflow = Workflow(name="bi_reporting", default_retry=RetryPolicy(max_retries=1))
ctx = Context(workflow_id=bi_workflow.workflow_id)

# Register steps (decorator-based in production; explicit here for clarity)
bi_workflow._steps = {}

bi_workflow.step(step_ingest_data.__wrapped__)(step_ingest_data) \
    if hasattr(step_ingest_data, "__wrapped__") else None

# Manual registration (mirrors what @step decorator does)
def _register(workflow, fn, input_type, output_type):
    workflow._steps[input_type.__name__] = {
        "fn": fn, "input_type": input_type, "output_type": output_type,
        "retry": RetryPolicy(max_retries=1), "timeout_s": None,
        "name": fn.__name__,
    }

# Fan-out: two steps both accept DataReadyEvent
# (we serialise for demo; production engine runs them concurrently)
def step_data_ready_fanout(ctx: Context, ev: DataReadyEvent) -> KPIsReadyEvent:
    "Runs both parallel branches sequentially and returns KPIsReadyEvent."
    step_retrieve_context(ctx, ev)   # writes to ctx, returns ContextReadyEvent
    return step_compute_kpis(ctx, ev)

_register(bi_workflow, step_ingest_data,       StartEvent, DataReadyEvent)
_register(bi_workflow, step_data_ready_fanout, DataReadyEvent, KPIsReadyEvent)
_register(bi_workflow, step_detect_anomalies,  KPIsReadyEvent, AnomalyReportEvent)
_register(bi_workflow, step_generate_insights, AnomalyReportEvent, InsightsReadyEvent)
_register(bi_workflow, step_format_report,     InsightsReadyEvent, StopEvent)

print("Workflow registered with steps:")
for name, meta in bi_workflow._steps.items():
    print(f"  {meta['name']:30s} accepts={name}")

In [ ]:
print("=" * 60)
print("RUNNING BI REPORTING WORKFLOW")
print("=" * 60)

start = StartEvent(config={"data_source": "simulated_sales_90d"})
stop = bi_workflow.run(start_event=start, ctx=ctx)

print()
print("WORKFLOW COMPLETE")
print(f"  Status:  {stop.status}")
if stop.result:
    r: ExecutiveReport = stop.result
    print(f"  Report:  {r.report_id}")
    print(f"  Period:  {r.period}")
    print(f"  KPIs:    ${r.kpis.total_revenue:,.0f} revenue, {r.kpis.total_orders:,} orders")
    print(f"  Anomalies:  {len(r.anomalies)} ({r.critical_anomaly_count} critical)")
    print(f"  Insights:   {len(r.insights)} ({len(r.high_confidence_insights)} high-confidence)")
    print(f"  Pipeline:   {r.total_pipeline_ms:.0f} ms")

## Execution Trace

The tracer captured every step's timing, input/output events, and retry count.

In [ ]:
trace_df = bi_workflow.tracer.report()
print("STEP EXECUTION TRACE")
print("=" * 75)
print(trace_df.to_string(index=False))
print(f"\nTotal traced time: {bi_workflow.tracer.total_ms:.1f} ms")

In [ ]:
fig = go.Figure()
colors = {"ok": "#2ecc71", "error": "#e74c3c", "retried": "#f39c12"}
for _, row in trace_df.iterrows():
    fig.add_trace(go.Bar(
        x=[row["duration_ms"]], y=[row["step"]],
        orientation="h", name=row["status"],
        marker_color=colors.get(row["status"], "#3498db"),
        showlegend=False,
        text=f'{row["duration_ms"]:.1f}ms',
        textposition="outside",
    ))
fig.update_layout(
    title="Workflow Step Duration (ms)",
    xaxis_title="Duration (ms)",
    template="plotly_white", height=350,
    margin=dict(l=180),
)
fig.show()

## Executive Report Output

The final structured `ExecutiveReport` validated by Pydantic.

In [ ]:
report: ExecutiveReport = ctx.get("report")

print("=" * 70)
print(f"EXECUTIVE REPORT  {report.report_id}")
print("=" * 70)
print(f"Generated: {report.generated_at}")
print(f"Period:    {report.period}")
print()
print("EXECUTIVE SUMMARY")
print("-" * 70)
for line in textwrap.wrap(report.executive_summary, width=70):
    print(" ", line)
print()
print("KEY PERFORMANCE INDICATORS")
print("-" * 70)
kpi_dict = report.kpis.model_dump()
for key, value in kpi_dict.items():
    labeled = key.replace("_", " ").title()
    if isinstance(value, float) and "pct" in key:
        print(f"  {labeled:<35s}: {value:+.2f}%")
    elif isinstance(value, float):
        fmted = f"${value:,.2f}" if "revenue" in key or "value" in key else f"{value:.4f}"
        print(f"  {labeled:<35s}: {fmted}")
    elif isinstance(value, int):
        print(f"  {labeled:<35s}: {value:,}")
    else:
        print(f"  {labeled:<35s}: {value}")

In [ ]:
print("\nANOMALIES DETECTED")
print("-" * 70)
if report.anomalies:
    for a in report.anomalies:
        sev_icon = {"critical": "🔴", "high": "🟠", "medium": "🟡", "low": "🟢"}.get(
            a.severity, "⚪")
        print(f"\n  [{a.severity.upper()}] {a.field}")
        for line in textwrap.wrap(a.description, width=65):
            print(f"    {line}")
        print(f"    Action: {a.recommended_action[:80]}")
else:
    print("  No anomalies detected.")

print("\nANALYST INSIGHTS")
print("-" * 70)
for i, ins in enumerate(report.insights, 1):
    print(f"\n  [{i}] [{ins.category.upper()}] {ins.headline}")
    print(f"       Confidence: {ins.confidence:.0%} | Priority: {ins.priority}")
    for line in textwrap.wrap(ins.body, width=65):
        print(f"       {line}")
    print(f"       Citations: {', '.join(ins.citations)}")

print(f"\nData sources used: {', '.join(report.data_sources)}")
print(f"Total pipeline: {report.total_pipeline_ms:.0f} ms")

## Visual Dashboard

In [ ]:
report: ExecutiveReport = ctx.get("report")
daily_rev: pd.DataFrame = ctx.get("daily_revenue")
region_df: pd.DataFrame = ctx.get("region_breakdown")
product_df: pd.DataFrame = ctx.get("product_breakdown")
channel_df: pd.DataFrame = ctx.get("channel_breakdown")

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        "Daily Revenue Trend (90d)",
        "Revenue by Region",
        "Revenue by Product",
        "Revenue by Channel",
        "Anomaly Severity Distribution",
        "Insight Confidence Scores",
    ],
    specs=[[{"type": "scatter"}, {"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "pie"}, {"type": "bar"}]],
)

# 1. Daily trend
fig.add_trace(
    go.Scatter(x=daily_rev["date"].astype(str),
               y=daily_rev["revenue"],
               mode="lines", name="Revenue",
               line=dict(color="#3498db", width=2)),
    row=1, col=1
)
# 7-day rolling avg
daily_rev["rolling7"] = daily_rev["revenue"].rolling(7).mean()
fig.add_trace(
    go.Scatter(x=daily_rev["date"].astype(str),
               y=daily_rev["rolling7"],
               mode="lines", name="7-day avg",
               line=dict(color="#e74c3c", dash="dot", width=1.5)),
    row=1, col=1
)

# 2. Region
fig.add_trace(
    go.Bar(x=region_df["region"], y=region_df["revenue"],
           marker_color="#2ecc71", showlegend=False),
    row=1, col=2
)

# 3. Product
fig.add_trace(
    go.Bar(x=product_df["product_category"], y=product_df["revenue"],
           marker_color="#9b59b6", showlegend=False),
    row=1, col=3
)

# 4. Channel
fig.add_trace(
    go.Bar(x=channel_df["channel"], y=channel_df["revenue"],
           marker_color="#f39c12", showlegend=False),
    row=2, col=1
)

# 5. Anomaly severity pie
severity_counts = {"critical": 0, "high": 0, "medium": 0, "low": 0}
for a in report.anomalies:
    severity_counts[a.severity] += 1
fig.add_trace(
    go.Pie(labels=list(severity_counts.keys()),
           values=list(severity_counts.values()),
           marker_colors=["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71"],
           showlegend=False),
    row=2, col=2
)

# 6. Insight confidence
ins_names = [f"INS-{i+1}: {ins.category}" for i, ins in enumerate(report.insights)]
ins_conf = [ins.confidence for ins in report.insights]
fig.add_trace(
    go.Bar(x=ins_names, y=ins_conf,
           marker_color=["#3498db" if c >= 0.85 else "#95a5a6" for c in ins_conf],
           showlegend=False),
    row=2, col=3
)

fig.update_layout(
    height=650, template="plotly_white",
    title_text="Executive BI Report Dashboard",
)
fig.show()

## Context Access Map

Visualises which steps wrote and which steps read each context key -- the "data lineage" of the workflow.

In [ ]:
access_log = ctx._access_log
df_access = pd.DataFrame(access_log)
df_access["ts_rel"] = df_access["ts"] - df_access["ts"].min()

# Pivot table: context key × step, colour = op type
write_ops = df_access[df_access["op"] == "set"].groupby(["key", "step"]).size().unstack(fill_value=0)
read_ops  = df_access[df_access["op"] == "get"].groupby(["key", "step"]).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if not write_ops.empty:
    sns.heatmap(write_ops, ax=axes[0], cmap="YlOrRd", annot=True, fmt="d",
                cbar_kws={"label": "Write count"})
    axes[0].set_title("Context Writes per Step")
    axes[0].set_xlabel("Step"); axes[0].set_ylabel("Context Key")
    axes[0].tick_params(axis="x", rotation=30)

if not read_ops.empty:
    sns.heatmap(read_ops, ax=axes[1], cmap="Blues", annot=True, fmt="d",
                cbar_kws={"label": "Read count"})
    axes[1].set_title("Context Reads per Step")
    axes[1].set_xlabel("Step"); axes[1].set_ylabel("Context Key")
    axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## Retrieval Evidence: What Context Was Used?

For every retrieved document, show which insight or anomaly it supported -- this is the **citation chain** that makes RAG outputs auditable.

In [ ]:
retrieved = ctx.get("retrieved_context") or []
print("RETRIEVAL EVIDENCE CHAIN")
print("=" * 75)
print(f"Total retrieved documents: {len(retrieved)}")
print()

# Citations used in insights
used_cites = set()
for ins in report.insights:
    used_cites.update(ins.citations)
for a in report.anomalies:
    pass   # anomalies reference in text

for doc in sorted(retrieved, key=lambda x: -x["score"]):
    used = "✓ CITED" if doc["doc_id"] in used_cites else "  unused"
    print(f"  {used}  [{doc['doc_id']}] {doc['title']}")
    print(f"          Score: {doc['score']:.4f}")
    print(f"          {doc['snippet'][:150]}...")
    print()

## Deep Dive: Parallel vs Sequential Execution

LlamaIndex Workflows support true async parallelism for independent steps. Here we time what the fan-out steps would cost sequentially vs concurrently.

In [ ]:
import threading

def time_sequential():
    t0 = time.time()
    fake_ctx = Context("benchmark")
    fake_ctx.set("raw_data", RAW_DATA, "bench")
    fake_ctx.set("date_range", ("2026-01-12", "2026-04-11"), "bench")
    ev = DataReadyEvent(row_count=len(RAW_DATA), column_count=9,
                        date_range=("2026-01-12", "2026-04-11"),
                        source_id="bench", validation_passed=True)
    step_retrieve_context(fake_ctx, ev)
    step_compute_kpis(fake_ctx, ev)
    return time.time() - t0


def time_simulated_parallel():
    """Simulates thread-level parallelism (async in real LlamaIndex)."""
    fake_ctx = Context("benchmark_par")
    fake_ctx.set("raw_data", RAW_DATA, "bench")
    fake_ctx.set("date_range", ("2026-01-12", "2026-04-11"), "bench")
    ev = DataReadyEvent(row_count=len(RAW_DATA), column_count=9,
                        date_range=("2026-01-12", "2026-04-11"),
                        source_id="bench", validation_passed=True)

    ctx_a = Context("par-a")
    ctx_b = Context("par-b")
    for k, v in fake_ctx._store.items():
        ctx_a.set(k, v, "copy"); ctx_b.set(k, v, "copy")

    t0 = time.time()
    t1 = threading.Thread(target=step_retrieve_context, args=(ctx_a, ev))
    t2 = threading.Thread(target=step_compute_kpis,     args=(ctx_b, ev))
    t1.start(); t2.start()
    t1.join();  t2.join()
    return time.time() - t0


reps = 5
seq_times = [time_sequential() for _ in range(reps)]
par_times  = [time_simulated_parallel() for _ in range(reps)]

seq_mean = np.mean(seq_times) * 1000
par_mean = np.mean(par_times) * 1000
speedup  = seq_mean / par_mean if par_mean > 0 else 1.0

print(f"Sequential (avg over {reps} runs): {seq_mean:.1f} ms")
print(f"Parallel   (avg over {reps} runs): {par_mean:.1f} ms")
print(f"Speedup:                           {speedup:.2f}×")
print()
print("Observation:")
if speedup > 1.05:
    print("  Parallel execution is faster -- confirms steps are independent.")
else:
    print("  Steps are fast enough that parallelism overhead dominates at this scale.")
    print("  At production LLM latency (300-2000ms per step),")
    print("  parallel fan-out would save the full sequential cost.")

## When to Use LlamaIndex Workflows

### Workflow vs Chain vs Agent

| Scenario | Best Pattern | Why |
|---|---|---|
| Fixed sequence of operations | **Simple Chain** | Lower overhead, easier to debug |
| Steps with shared state across multiple retrievals | **Workflow** | Context object prevents parameter bloat |
| Two independent steps that can run in parallel | **Workflow (fan-out)** | Parallel execution saves wall-clock time |
| Wait for multiple parallel results before proceeding | **Workflow (join)** | Built-in event join gate |
| Dynamic tool selection based on query | **Agent** | Agents decide which tools to call at runtime |
| Structured output validation at each step | **Workflow** | Pydantic schema enforcement per step |
| Long-running pipeline with retries | **Workflow** | Per-step retry policies and error isolation |
| Simple RAG query-answer | **Query Engine** | Simpler than a full workflow for single-step RAG |

### Workflow Design Checklist

```
✓ Does each step have a single, clear responsibility?
✓ Is all shared state in the Context (not event payloads)?
✓ Are event schemas minimal -- just what the next step needs?
✓ Have you added Pydantic validators for outputs at trust boundaries?
✓ Is each step independently testable with a mock Context?
✓ Have you identified which steps could run in parallel?
✓ Is the retrieval step triggered once per workflow, not per step?
```

## Retry & Fault Tolerance Demo

Demonstrate that the retry policy transparently handles transient failures.

In [ ]:
fail_count = [0]   # mutable container so inner function can mutate it

def unreliable_step(ctx: Context, ev: StartEvent) -> StopEvent:
    """Fails the first 2 times, succeeds on attempt 3."""
    fail_count[0] += 1
    if fail_count[0] <= 2:
        raise RuntimeError(f"Transient failure (attempt {fail_count[0]})")
    return StopEvent(result=f"succeeded on attempt {fail_count[0]}", status="success")


retry_workflow = Workflow(
    name="retry_demo",
    default_retry=RetryPolicy(max_retries=3, base_delay_s=0.01),
)
retry_workflow._steps["StartEvent"] = {
    "fn": unreliable_step, "input_type": StartEvent,
    "output_type": StopEvent,
    "retry": RetryPolicy(max_retries=3, base_delay_s=0.01),
    "timeout_s": None, "name": "unreliable_step",
}

print("Running retry demo (step fails twice then succeeds)...")
result = retry_workflow.run(StartEvent())
print(f"Result: {result.result}")
print(f"Retry count: {fail_count[0] - 1} retries before success")

trace = retry_workflow.tracer.report()
print(trace.to_string(index=False))

## Structured Output Validation Demo

Pydantic catches invalid outputs at the model boundary before they can propagate downstream.

In [ ]:
print("Testing Pydantic validation on SalesKPIs:")
print()

# ── Valid case ────────────────────────────────────────────────────────
try:
    kpi = SalesKPIs(
        period="Q1 2026", total_revenue=1_234_567.89, total_orders=13_000,
        avg_order_value=95.04, revenue_growth_pct=8.2, order_growth_pct=5.1,
        top_region="North America", top_product="Electronics",
        conversion_rate=0.028, return_rate=0.085, csat_score=3.95,
    )
    print(f"  ✓ Valid KPIs accepted: csat={kpi.csat_score}, return_rate={kpi.return_rate}")
except Exception as e:
    print(f"  ✗ Unexpected error: {e}")

# ── Invalid return_rate (> 1.0) ───────────────────────────────────────
try:
    bad1 = SalesKPIs(
        period="Q1", total_revenue=1000, total_orders=10,
        avg_order_value=100, revenue_growth_pct=0, order_growth_pct=0,
        top_region="NA", top_product="Elec",
        conversion_rate=0.03, return_rate=1.5,   # INVALID
        csat_score=4.0,
    )
    print(f"  ✗ Should have rejected return_rate=1.5")
except Exception as e:
    print(f"  ✓ Rejected return_rate=1.5: {type(e).__name__}")

# ── Invalid csat_score (> 5.0) ────────────────────────────────────────
try:
    bad2 = SalesKPIs(
        period="Q1", total_revenue=1000, total_orders=10,
        avg_order_value=100, revenue_growth_pct=0, order_growth_pct=0,
        top_region="NA", top_product="Elec",
        conversion_rate=0.03, return_rate=0.09,
        csat_score=7.2,   # INVALID
    )
    print(f"  ✗ Should have rejected csat_score=7.2")
except Exception as e:
    print(f"  ✓ Rejected csat_score=7.2: {type(e).__name__}")

# ── Invalid anomaly severity ──────────────────────────────────────────
try:
    bad3 = Anomaly(
        field="test", description="test", severity="extreme",   # INVALID
        value=0.5, expected_range=(0, 1), evidence="test",
        recommended_action="test",
    )
    print(f"  ✗ Should have rejected severity='extreme'")
except Exception as e:
    print(f"  ✓ Rejected severity='extreme': {type(e).__name__}")

print()
print("All validation tests passed.")

## Key Takeaways

### What We Built
- **5-step BI reporting workflow**: ingest → (retrieve + KPIs in parallel) → anomalies → insights → report
- **Typed event bus**: every step transition is a validated Pydantic model
- **Shared Context**: eliminates passing large state dicts through function signatures
- **Knowledge base retrieval**: TF-IDF contextual retrieval that mirrors LlamaIndex's `VectorStoreIndex`
- **Structured outputs**: Pydantic-validated `SalesKPIs`, `Anomaly`, `AnalystInsight`, `ExecutiveReport`
- **Step-level retry**: transparent exponential backoff with configurable max attempts
- **Tracing**: per-step timing, event log, and access map for full observability
- **Parallel fan-out**: benchmarked sequential vs parallel step execution

### Core Concepts Summary

| Concept | This Notebook | LlamaIndex Production |
|---|---|---|
| `Event` | Pydantic BaseModel | `llama_index.core.workflow.Event` |
| `Context` | Dict-backed store with access log | `llama_index.core.workflow.Context` |
| `@step` decorator | Manual `_register()` | `@step` on async coroutines |
| `Workflow.run()` | Sync event loop | `await workflow.run()` |
| `StartEvent` | Carries config | `llama_index.core.workflow.StartEvent` |
| `StopEvent` | Carries final result | `llama_index.core.workflow.StopEvent` |
| Fan-out | Serialised in demo | True async coroutine concurrency |
| Join gate | Single step reads ctx | Multi-event step signature |

### Production Extensions
1. Replace `MockLLM` with `llama_index.llms.OpenAI` / `Ollama` / `Anthropic`
2. Replace `KnowledgeBase` with `VectorStoreIndex` backed by Chroma or Pinecone
3. Add `async/await` to all steps for true concurrency
4. Plug in OpenTelemetry exporter to the tracer for Jaeger/Grafana observability
5. Add `HumanInputEvent` step for human-in-the-loop approval of anomaly alerts

---
*Educational notebook -- part of the NLP / Retrieval-Augmented Generation portfolio.*